
# 07. Generación de embeddings para búsqueda semántica

## Communication Intelligence Assistant

### Objetivo

Implementar un proceso de vectorización semántica de noticias mediante un modelo de embeddings multilingüe, con el propósito de representar su contenido textual como vectores numéricos y habilitar su posterior recuperación por similitud semántica.

### Descripción

Este notebook desarrolla la etapa de generación de embeddings del sistema *Communication Intelligence Assistant*, utilizando noticias previamente adquiridas, procesadas y almacenadas en PostgreSQL.

Se empleará el modelo `intfloat/multilingual-e5-small`, basado en la arquitectura Transformer, para generar representaciones vectoriales del contenido textual en español.

Los embeddings resultantes se almacenarán mediante la extensión `pgvector` de PostgreSQL, conservando la trazabilidad entre los fragmentos procesados y sus noticias originales.

### Metodología

El procesamiento se organiza en las siguientes etapas:

1. **Configuración del entorno:** importación de dependencias y carga del modelo de embeddings.
2. **Validación inicial:** generación y verificación de embeddings de prueba.
3. **Extracción de datos:** recuperación de noticias desde PostgreSQL.
4. **Fragmentación textual:** división del contenido en fragmentos adecuados para el modelo.
5. **Vectorización:** generación de embeddings normalizados.
6. **Persistencia:** almacenamiento de fragmentos y vectores mediante pgvector.
7. **Validación:** comprobación de dimensiones, integridad y trazabilidad de los embeddings generados.

### Tecnologías utilizadas

- **Python:** implementación del procesamiento.
- **Sentence Transformers:** carga y ejecución del modelo de embeddings.
- **Multilingual E5 Small:** generación de representaciones vectoriales de 384 dimensiones.
- **PostgreSQL:** almacenamiento de noticias y metadatos.
- **pgvector:** almacenamiento y consulta de vectores.

### Resultado esperado

Obtener un conjunto de embeddings asociados a fragmentos de noticias, preparados para implementar un mecanismo de búsqueda semántica y recuperación aumentada por generación (RAG).

### Consideraciones

- Se conservarán los textos originales sin modificaciones.
- Se utilizará el prefijo `passage:` para codificar los fragmentos de noticias.
- Los vectores se normalizarán para facilitar las comparaciones mediante similitud coseno.
- Se documentarán los parámetros de fragmentación y vectorización para garantizar la reproducibilidad.
- La calidad de recuperación semántica se evaluará en una etapa posterior.

---



## 1. Configuración del entorno y carga del modelo

En esta sección se inicializa el modelo de embeddings que se utilizará para generar representaciones vectoriales de los textos periodísticos.

Se selecciona `intfloat/multilingual-e5-small`, un modelo multilingüe basado en Transformers, diseñado para tareas de recuperación semántica.

**Características principales:**

- **Modelo:** `intfloat/multilingual-e5-small`.
- **Dimensionalidad:** 384.
- **Longitud máxima de entrada:** 512 tokens.
- **Idioma de trabajo:** español.
- **Librería:** Sentence Transformers.

El modelo se ejecutará localmente, sin necesidad de utilizar servicios externos de inferencia.

Se verificará su correcta inicialización antes de comenzar el procesamiento del corpus.


In [ ]:
#pip install sentence-transformers

In [4]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv


In [5]:
# Configuración del modelo de embeddings.
MODEL_NAME = "intfloat/multilingual-e5-small"
NORMALIZE_EMBEDDINGS = True

# Carga del modelo preentrenado.
model = SentenceTransformer(MODEL_NAME)

# Verificación de las características del modelo.
embedding_dimension = model.get_sentence_embedding_dimension()
max_sequence_length = model.max_seq_length

print(f"Modelo cargado: {MODEL_NAME}")
print(f"Dimensiones del embedding: {embedding_dimension}")
print(f"Longitud máxima de entrada: {max_sequence_length}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo cargado: intfloat/multilingual-e5-small
Dimensiones del embedding: 384
Longitud máxima de entrada: 512


C:\Users\enzor\AppData\Local\Temp\ipykernel_31340\1752156372.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()



## 2. Generación y validación de embeddings

En esta sección se generan embeddings a partir de textos de prueba utilizando el modelo `intfloat/multilingual-e5-small`.

El objetivo es verificar que las representaciones vectoriales tengan la dimensionalidad esperada y se encuentren correctamente normalizadas.

El modelo E5 utiliza prefijos específicos según la función del texto:

- `passage:` para documentos o fragmentos que serán indexados.
- `query:` para consultas utilizadas durante la recuperación semántica.

Se utilizará la normalización L2 para obtener vectores de norma unitaria, adecuados para posteriores operaciones de similitud coseno.

Esta validación permite comprobar el funcionamiento de la vectorización antes de procesar el corpus completo de noticias.


In [6]:
# Textos de prueba relacionados con el dominio del proyecto.
sample_texts = [
    "passage: La inteligencia artificial transforma el mercado laboral.",
    "passage: Los avances científicos mejoran los tratamientos médicos.",
    "passage: Las empresas tecnológicas desarrollan nuevos modelos de IA.",
]

# Generación de embeddings normalizados.
sample_embeddings = model.encode(
    sample_texts,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    convert_to_numpy=True,
    show_progress_bar=False,
)

# Verificación de dimensiones y normalización.
embedding_norms = np.linalg.norm(sample_embeddings, axis=1)

print(f"Cantidad de textos: {len(sample_texts)}")
print(f"Dimensiones de la matriz: {sample_embeddings.shape}")
print(f"Normas L2: {embedding_norms}")

assert sample_embeddings.shape == (
    len(sample_texts),
    embedding_dimension,
), "La dimensionalidad de los embeddings no es la esperada."

assert np.allclose(embedding_norms, 1.0, atol=1e-5), (
    "Los embeddings no están correctamente normalizados."
)


Cantidad de textos: 3
Dimensiones de la matriz: (3, 384)
Normas L2: [1. 1. 1.]



### 2.1. Interpretación de resultados

La generación de embeddings permite verificar dos propiedades fundamentales:

1. **Dimensionalidad:** cada texto se representa mediante un vector de 384 componentes, de acuerdo con la arquitectura del modelo seleccionado.
2. **Normalización:** los vectores presentan una norma L2 aproximadamente igual a 1, lo que permite comparar sus direcciones mediante similitud coseno.

Los resultados confirman el funcionamiento técnico del proceso de vectorización. Sin embargo, esta comprobación no permite determinar todavía la calidad semántica del modelo.

La evaluación de la capacidad de recuperación se realizará posteriormente utilizando consultas y noticias.



## 3. Extracción del corpus de noticias

En esta sección se recuperan las noticias almacenadas en PostgreSQL para preparar el corpus que será utilizado durante la generación de embeddings.

Se utiliza la tabla `rss_news`, que contiene los textos periodísticos previamente procesados durante las etapas de adquisición y preparación de datos.

Para cada noticia se recuperan:

- **rss_news_id:** identificador único de la noticia.
- **title:** título de la publicación.
- **content:** contenido textual utilizado como fuente principal para la vectorización.
- **published_at:** fecha de publicación.
- **url:** enlace a la noticia original.

Se excluirán del procesamiento los registros cuyo contenido sea nulo o esté compuesto únicamente por espacios en blanco.

La extracción se realizará mediante una consulta de solo lectura, conservando la integridad de los datos originales.

### Resultado esperado

Obtener un DataFrame con las noticias disponibles para su posterior fragmentación y vectorización, manteniendo la trazabilidad mediante `rss_news_id`.


### Configuración del entorno experimental

Se configura la ruta raíz del proyecto para reutilizar
los módulos originales del asistente desde el directorio
de evaluación experimental.

In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name in {"notebooks", "evaluacion_ida"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [8]:
load_dotenv(
    PROJECT_ROOT / ".env.ida_caso_02",
    override=True,
)

True

In [9]:
EXPECTED_DATABASE = "ida_caso_02"
ENV_PATH = PROJECT_ROOT / ".env.ida_caso_02"

# Cargar las variables del entorno experimental.
if not load_dotenv(ENV_PATH, override=True):
    raise RuntimeError("No se pudo cargar el entorno experimental.")

In [11]:
from src.database.connection import get_connection
# Verificar la conexión con la base experimental.
with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT current_database();")
        database = cursor.fetchone()[0]

assert database == EXPECTED_DATABASE, (
    f"Base de datos incorrecta: {database}"
)

print(f"Base experimental activa: {database}")

Base experimental activa: ida_caso_02


In [12]:
from src.database.connection import get_connection

QUERY_NEWS = """
    SELECT
        rss_news_id,
        title,
        content,
        published_at,
        url
    FROM rss_news
    WHERE
        content IS NOT NULL
        AND BTRIM(content) <> ''
    ORDER BY rss_news_id;
"""

with get_connection() as connection:
    news_df = pd.read_sql_query(
        QUERY_NEWS,
        connection,
    )

print(f"Noticias recuperadas: {len(news_df)}")
print(f"Noticias únicas: {news_df['rss_news_id'].nunique()}")

display(news_df.head())


Noticias recuperadas: 50
Noticias únicas: 50


C:\Users\enzor\AppData\Local\Temp\ipykernel_31340\2009839109.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  news_df = pd.read_sql_query(


,rss_news_id,title,content,published_at,url
0,1,Energía Costa Serena: amplía la generación solar,La puesta en marcha de 24 MW adicionales fue c...,2026-04-01 10:00:00+00:00,https://caso02.example.invalid/noticias/001
1,2,Energía Costa Serena: reduce interrupciones el...,La mejora del servicio benefició a 800 hogares...,2026-04-02 10:00:00+00:00,https://caso02.example.invalid/noticias/002
2,3,Energía Costa Serena: recupera capacidad de un...,La reparación permitió recuperar 18 MW y refor...,2026-04-03 10:00:00+00:00,https://caso02.example.invalid/noticias/003
3,4,Energía Costa Serena: sufre una falla en una s...,El corte afectó a 1200 usuarios y generó recla...,2026-04-04 10:00:00+00:00,https://caso02.example.invalid/noticias/004
4,5,Energía Costa Serena: demora obras de transpor...,El retraso de tres meses agravó la incertidumb...,2026-04-05 10:00:00+00:00,https://caso02.example.invalid/noticias/005


In [13]:

assert not news_df.empty, "No se encontraron noticias para procesar."

assert news_df["rss_news_id"].is_unique, (
    "Se encontraron identificadores de noticias duplicados."
)

assert news_df["content"].notna().all(), (
    "Existen noticias con contenido nulo."
)

assert news_df["content"].str.strip().ne("").all(), (
    "Existen noticias con contenido vacío."
)

news_df["content_length"] = news_df["content"].str.len()

display(news_df["content_length"].describe())


count     50.000000
mean     807.760000
std       47.193289
min      724.000000
25%      780.000000
50%      800.500000
75%      822.750000
max      970.000000
Name: content_length, dtype: float64


## 4. Análisis de longitud y fragmentación del corpus

Los modelos de embeddings presentan restricciones sobre la cantidad máxima de tokens que pueden procesar en una única entrada.

El modelo `intfloat/multilingual-e5-small` admite un máximo de 512 tokens por secuencia. Por este motivo, resulta necesario analizar la longitud de las noticias antes de definir una estrategia de fragmentación (*chunking*).

### 4.1. Análisis de longitud en tokens

Se utilizará el tokenizador asociado al modelo para calcular la cantidad de tokens de cada noticia.

Este análisis permitirá:

- Identificar noticias que superan el límite del modelo.
- Estimar la necesidad de fragmentación.
- Fundamentar la selección del tamaño de los fragmentos.
- Evitar el truncamiento involuntario de contenido.

El conteo incluirá el prefijo `passage:` y los tokens especiales del modelo.

La estrategia de fragmentación se definirá a partir de estos resultados.


In [14]:

# Recuperación del tokenizador del modelo.
tokenizer = model.tokenizer

# Conteo de tokens sin truncamiento.
def count_tokens(text: str) -> int:
    """Cuenta los tokens de un texto utilizando el tokenizador E5."""
    return len(
        tokenizer.encode(
            f"passage: {text}",
            add_special_tokens=True,
            truncation=False,
        )
    )


news_df["token_count"] = news_df["content"].apply(count_tokens)

# Estadísticas descriptivas.
display(news_df["token_count"].describe())

# Noticias que superan el límite del modelo.
news_over_limit = (
    news_df["token_count"] > max_sequence_length
).sum()

print(f"Noticias analizadas: {len(news_df)}")
print(f"Noticias que superan {max_sequence_length} tokens: {news_over_limit}")
print(f"Porcentaje: {news_over_limit / len(news_df):.2%}")


count     50.000000
mean     150.560000
std        9.814941
min      134.000000
25%      145.500000
50%      149.500000
75%      154.000000
max      187.000000
Name: token_count, dtype: float64

Noticias analizadas: 50
Noticias que superan 512 tokens: 0
Porcentaje: 0.00%



### 4.2. Estrategia de fragmentación textual

El análisis del corpus identificó que 136 de las 186 noticias (73,12 %) superan el límite de 512 tokens admitido por el modelo.

Para evitar la pérdida de información por truncamiento, se implementará una estrategia de fragmentación mediante ventanas deslizantes (*sliding window*).

Se utilizarán inicialmente los siguientes parámetros:

- **Tamaño máximo:** 384 tokens por fragmento de contenido.
- **Solapamiento:** 64 tokens entre fragmentos consecutivos.
- **Contextualización:** inclusión del título de la noticia en cada fragmento.
- **Trazabilidad:** asociación de cada fragmento con el identificador de su noticia original.

Los parámetros seleccionados constituyen una configuración inicial que podrá evaluarse y ajustarse posteriormente.

La fragmentación se realizará utilizando el tokenizador del modelo para controlar la longitud efectiva de las entradas, considerando el prefijo `passage:` y los tokens especiales.

El objetivo es preservar el contenido original, mantener el contexto entre fragmentos y generar entradas compatibles con el modelo de embeddings.


In [15]:

CHUNK_SIZE = 384
CHUNK_OVERLAP = 64
PASSAGE_PREFIX = "passage: "


def split_news_into_chunks(
    news_id: int,
    title: str,
    content: str,
) -> list[dict]:
    """Divide una noticia en fragmentos compatibles con el modelo E5."""
    if not 0 <= CHUNK_OVERLAP < CHUNK_SIZE:
        raise ValueError("La configuración de chunking no es válida.")

    title = title or ""
    content = content or ""

    # Tokenización sin agregar tokens especiales.
    content_tokens = tokenizer.encode(
        content,
        add_special_tokens=False,
        truncation=False,
    )

    # Reservar espacio para el prefijo, el título y tokens especiales.
    header = f"{PASSAGE_PREFIX}{title}\n"
    header_tokens = tokenizer.encode(
        header,
        add_special_tokens=False,
    )

    special_tokens = tokenizer.num_special_tokens_to_add(pair=False)

    available_tokens = min(
        CHUNK_SIZE,
        max_sequence_length - len(header_tokens) - special_tokens,
    )

    if available_tokens <= CHUNK_OVERLAP:
        raise ValueError("El título no deja espacio suficiente para el contenido.")

    step = available_tokens - CHUNK_OVERLAP
    chunks = []

    for chunk_index, start in enumerate(range(0, len(content_tokens), step)):
        token_window = content_tokens[start:start + available_tokens]

        if not token_window:
            break

        chunk_text = tokenizer.decode(
            token_window,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        embedding_text = f"{header}{chunk_text}"

        # Validación sobre el texto efectivamente reconstruido.
        actual_tokens = tokenizer.encode(
            embedding_text,
            add_special_tokens=True,
            truncation=False,
        )

        if len(actual_tokens) > max_sequence_length:
            raise ValueError(
                f"El fragmento {chunk_index} de la noticia "
                f"{news_id} supera el límite del modelo."
            )

        chunks.append(
            {
                "rss_news_id": news_id,
                "chunk_index": chunk_index,
                "chunk_text": chunk_text,
                "embedding_text": embedding_text,
                "token_count": len(actual_tokens),
            }
        )

        if start + available_tokens >= len(content_tokens):
            break

    return chunks


In [16]:

sample_news = news_df.loc[
    news_df["rss_news_id"] == 3
].iloc[0]

sample_chunks = split_news_into_chunks(
    news_id=int(sample_news["rss_news_id"]),
    title=sample_news["title"],
    content=sample_news["content"],
)

print(f"Noticia: {sample_news['title']}")
print(f"Fragmentos generados: {len(sample_chunks)}")

for chunk in sample_chunks:
    print(
        f"Chunk {chunk['chunk_index']}: "
        f"{chunk['token_count']} tokens"
    )

assert all(
    chunk["token_count"] <= max_sequence_length
    for chunk in sample_chunks
), "Se generaron fragmentos que superan el límite del modelo."


Noticia: Energía Costa Serena: recupera capacidad de una central
Fragmentos generados: 1
Chunk 0: 179 tokens



### 4.3. Fragmentación y validación del corpus completo

Una vez verificado el funcionamiento de la estrategia de fragmentación sobre una noticia individual, se aplica el procedimiento al conjunto completo de noticias.

Cada fragmento conserva el identificador de la noticia original y un índice que permite reconstruir su orden.

Se realizarán las siguientes verificaciones:

- Cantidad total de noticias procesadas y fragmentos generados.
- Unicidad de la combinación `rss_news_id` y `chunk_index`.
- Cumplimiento del límite máximo de 512 tokens.
- Ausencia de fragmentos vacíos.
- Distribución de la longitud de los fragmentos.

Estas verificaciones permiten evaluar la integridad técnica del proceso antes de generar los embeddings.


In [17]:

# Generación de fragmentos para todas las noticias.
all_chunks = []

for news in news_df.itertuples(index=False):
    news_chunks = split_news_into_chunks(
        news_id=int(news.rss_news_id),
        title=news.title,
        content=news.content,
    )

    all_chunks.extend(news_chunks)

# Construcción del DataFrame de fragmentos.
chunks_df = pd.DataFrame(all_chunks)

print(f"Noticias originales: {len(news_df)}")
print(f"Noticias fragmentadas: {chunks_df['rss_news_id'].nunique()}")
print(f"Fragmentos generados: {len(chunks_df)}")

display(chunks_df.head())


Noticias originales: 50
Noticias fragmentadas: 50
Fragmentos generados: 50


,rss_news_id,chunk_index,chunk_text,embedding_text,token_count
0,1,0,La puesta en marcha de 24 MW adicionales fue c...,passage: Energía Costa Serena: amplía la gener...,199
1,2,0,La mejora del servicio benefició a 800 hogares...,passage: Energía Costa Serena: reduce interrup...,195
2,3,0,La reparación permitió recuperar 18 MW y refor...,passage: Energía Costa Serena: recupera capaci...,179
3,4,0,El corte afectó a 1200 usuarios y generó recla...,passage: Energía Costa Serena: sufre una falla...,171
4,5,0,El retraso de tres meses agravó la incertidumb...,passage: Energía Costa Serena: demora obras de...,168


In [18]:

# Verificar que todas las noticias estén representadas.
assert set(chunks_df["rss_news_id"]) == set(news_df["rss_news_id"]), (
    "Existen noticias sin fragmentos."
)

# Verificar identificadores únicos por fragmento.
assert not chunks_df.duplicated(
    subset=["rss_news_id", "chunk_index"]
).any(), "Se encontraron fragmentos duplicados."

# Verificar límites del modelo.
assert chunks_df["token_count"].le(max_sequence_length).all(), (
    "Existen fragmentos que superan el límite del modelo."
)

# Verificar que no existan fragmentos vacíos.
assert chunks_df["chunk_text"].str.strip().ne("").all(), (
    "Existen fragmentos vacíos."
)

# Estadísticas de fragmentación.
chunks_per_news = chunks_df.groupby("rss_news_id").size()

print("Estadísticas de tokens por fragmento:")
display(chunks_df["token_count"].describe())

print("Estadísticas de fragmentos por noticia:")
display(chunks_per_news.describe())

print("Validaciones completadas correctamente.")


Estadísticas de tokens por fragmento:


count     50.000000
mean     160.580000
std       11.373241
min      139.000000
25%      153.000000
50%      161.000000
75%      165.500000
max      199.000000
Name: token_count, dtype: float64

Estadísticas de fragmentos por noticia:


count    50.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0
dtype: float64

Validaciones completadas correctamente.



## 5. Generación de embeddings

En esta etapa se generan las representaciones vectoriales de los fragmentos obtenidos durante el procesamiento del corpus.

Se utiliza el modelo `intfloat/multilingual-e5-small`, que transforma cada entrada textual en un vector de 384 dimensiones.

### Estrategia de procesamiento

La generación se realizará mediante procesamiento por lotes (*batch processing*) para controlar el uso de memoria y facilitar la ejecución local.

Se establecen los siguientes parámetros:

- **Modelo:** `intfloat/multilingual-e5-small`.
- **Cantidad de fragmentos:** 550.
- **Dimensionalidad:** 384.
- **Tamaño de lote inicial:** 32 fragmentos.
- **Normalización:** L2.
- **Entrada:** título de la noticia y contenido del fragmento, precedidos por `passage:`.

El objetivo es obtener una matriz de embeddings normalizados, manteniendo el orden de los registros del DataFrame.

La persistencia en PostgreSQL se realizará posteriormente, una vez verificada la integridad de los vectores generados.


In [19]:

import time

# Configuración del procesamiento.
BATCH_SIZE = 32

# Preparación de los textos.
embedding_texts = chunks_df["embedding_text"].tolist()

# Generación de embeddings.
start_time = time.perf_counter()

embeddings = model.encode(
    embedding_texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=NORMALIZE_EMBEDDINGS,
    convert_to_numpy=True,
    show_progress_bar=True,
)

elapsed_time = time.perf_counter() - start_time

print(f"Fragmentos procesados: {len(embedding_texts)}")
print(f"Dimensiones de la matriz: {embeddings.shape}")
print(f"Tiempo de procesamiento: {elapsed_time:.2f} segundos")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Fragmentos procesados: 50
Dimensiones de la matriz: (50, 384)
Tiempo de procesamiento: 1.67 segundos


In [20]:

# Verificar cantidad y dimensionalidad.
assert embeddings.shape == (
    len(chunks_df),
    embedding_dimension,
), "La matriz de embeddings tiene dimensiones incorrectas."

# Verificar ausencia de valores inválidos.
assert np.isfinite(embeddings).all(), (
    "Se encontraron valores NaN o infinitos."
)

# Verificar normalización L2.
embedding_norms = np.linalg.norm(embeddings, axis=1)

assert np.allclose(embedding_norms, 1.0, atol=1e-5), (
    "Se encontraron embeddings sin normalizar."
)

print("Validaciones de embeddings completadas correctamente.")
print(f"Tipo de dato: {embeddings.dtype}")
print(f"Norma L2 mínima: {embedding_norms.min():.6f}")
print(f"Norma L2 máxima: {embedding_norms.max():.6f}")


Validaciones de embeddings completadas correctamente.
Tipo de dato: float32
Norma L2 mínima: 1.000000
Norma L2 máxima: 1.000000



## 6. Persistencia de embeddings en PostgreSQL

Una vez generados y validados los embeddings, se procede a preparar su almacenamiento en PostgreSQL mediante la extensión `pgvector`.

### Diseño de almacenamiento

Se utilizará una tabla independiente denominada `rss_news_chunks`, relacionada con la tabla `rss_news` mediante una clave foránea.

Cada registro representará un fragmento textual e incluirá:

- Identificador único del fragmento.
- Identificador de la noticia original.
- Posición del fragmento dentro de la noticia.
- Texto utilizado para la vectorización.
- Representación vectorial de 384 dimensiones.
- Nombre del modelo utilizado.
- Fecha de creación del registro.

### Consideraciones de diseño

- Se preservan los registros originales de `rss_news`.
- Se mantiene la trazabilidad entre noticias y fragmentos.
- Se utiliza el tipo `vector(384)` de pgvector.
- Se establece una restricción de unicidad para evitar duplicados.
- Se permite identificar el modelo utilizado para generar cada embedding.
- La persistencia se implementará mediante SQL parametrizado y transacciones.

El almacenamiento permitirá implementar posteriormente consultas de similitud semántica sobre el corpus.


In [21]:

from pgvector.psycopg import register_vector

INSERT_CHUNKS_QUERY = """
    INSERT INTO rss_news_chunks (
        fk_rss_news_id,
        chunk_index,
        chunk_text,
        embedding,
        model_name
    )
    VALUES (%s, %s, %s, %s, %s)
    ON CONFLICT (fk_rss_news_id, chunk_index, model_name)
    DO NOTHING;
"""

# Preparación de registros conservando el orden de los embeddings.
records = [
    (
        int(chunk.rss_news_id),
        int(chunk.chunk_index),
        chunk.chunk_text,
        embeddings[index],
        MODEL_NAME,
    )
    for index, chunk in enumerate(
        chunks_df.itertuples(index=False)
    )
]

# Inserción mediante una transacción.
with get_connection() as connection:
    register_vector(connection)

    with connection.cursor() as cursor:
        cursor.executemany(
            INSERT_CHUNKS_QUERY,
            records,
        )

print(f"Registros preparados para inserción: {len(records)}")
print("Operación de persistencia finalizada.")


Registros preparados para inserción: 50
Operación de persistencia finalizada.


In [22]:
query = """
    SELECT
        COUNT(*) AS total_chunks,
        COUNT(DISTINCT fk_rss_news_id) AS total_news,
        MIN(vector_dims(embedding)) AS min_dimensions,
        MAX(vector_dims(embedding)) AS max_dimensions,
        COUNT(*) FILTER (
            WHERE embedding IS NULL
        ) AS null_embeddings
    FROM rss_news_chunks;
"""

with get_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)

        embeddings_validation_df = pd.DataFrame(
            cursor.fetchall(),
            columns=[column.name for column in cursor.description],
        )

display(embeddings_validation_df)

,total_chunks,total_news,min_dimensions,max_dimensions,null_embeddings
0,50,50,384,384,0
